### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 업스테이지의 콘텐츠 자산으로, 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.

# 실습 (심화) — Time/Token Budget·Sandbox 실행·통합 Reliability Layer
**Production-Grade Agent Engineering · Colab 실습**

> 목표: 이전 실습의 회로 차단기·재시도·멱등성을 **하나의 도구 호출 경로로 통합**하고, 강의가 강조한 **① 시간/토큰 예산 강제**와 **② 샌드박스 코드 실행(보안)**을 직접 구현합니다. (LLM 키 불필요 — 순수 로직)

## 0. 셋업
이 실습은 외부 API 없이 표준 라이브러리만 사용합니다.

In [ ]:
import time, random, signal
print('준비 완료')

## 1. Time Budget — 시간 초과 강제 종료
블록 실행 시간을 제한합니다. (Colab/Unix 기준 `signal.SIGALRM` 사용)

In [ ]:
class TimeBudgetExceeded(Exception):
    pass

class time_budget:
    # with 블록의 실행 시간을 limit 초로 제한 (Unix/Colab 메인스레드)
    def __init__(self, limit):
        self.limit = limit
    def __enter__(self):
        def _raise(signum, frame):
            raise TimeBudgetExceeded(f"⏱ {self.limit}s 초과")
        self._old = signal.signal(signal.SIGALRM, _raise)
        signal.setitimer(signal.ITIMER_REAL, self.limit)
        return self
    def __exit__(self, *a):
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, self._old)
        return False

try:
    with time_budget(1.0):
        time.sleep(3)            # 3초 작업 → 1초에서 중단
except TimeBudgetExceeded as e:
    print("가드 작동:", e)

## 2. Token Budget — 예산 소진 시 중단
에이전트 루프에서 호출마다 토큰을 차감하고, 예산이 바닥나면 다음 단계를 막습니다.

In [ ]:
class BudgetExceeded(Exception):
    pass

class Budget:
    def __init__(self, max_tokens, max_steps):
        self.max_tokens, self.max_steps = max_tokens, max_steps
        self.tokens = self.steps = 0
    def charge(self, tokens):
        self.steps += 1; self.tokens += tokens
        if self.steps > self.max_steps:
            raise BudgetExceeded(f"스텝 {self.steps}>{self.max_steps}")
        if self.tokens > self.max_tokens:
            raise BudgetExceeded(f"토큰 {self.tokens}>{self.max_tokens}")

def mock_llm(prompt_tokens, output_tokens=80):
    return prompt_tokens + output_tokens     # 실제 호출 대신 토큰만 계산

budget = Budget(max_tokens=500, max_steps=10)
try:
    for step in range(10):
        used = mock_llm(prompt_tokens=120)
        budget.charge(used)
        print(f"  step{step+1}: +{used}토큰 (누적 {budget.tokens})")
except BudgetExceeded as e:
    print("가드 작동:", e, "→ 루프 중단")

## 3. 샌드박스 실행 — 위험 코드 차단
에이전트가 생성한 코드를 **제한된 빌트인**으로만 실행해 파일 삭제·임의 import 같은 위험을 막고, 무한 루프는 시간 예산으로 끊습니다.

In [ ]:
_B = __builtins__ if isinstance(__builtins__, dict) else vars(__builtins__)
SAFE_BUILTINS = {k: _B[k] for k in
    ["range","len","sum","min","max","print","abs","sorted","list","dict","int","float","str"]}

def run_in_sandbox(code, limit=2.0):
    g = {"__builtins__": SAFE_BUILTINS}
    try:
        with time_budget(limit):
            exec(code, g)
        return g.get("result", "(result 없음)")
    except TimeBudgetExceeded:
        return "🛑 시간 초과(무한 루프 의심) — 중단"
    except Exception as e:
        return f"🛑 차단됨: {type(e).__name__}: {e}"

print("안전 코드:", run_in_sandbox("result = sum(range(10))"))
print("파일 삭제 시도:", run_in_sandbox("import os; os.remove('/etc/hosts')"))
print("무한 루프:", run_in_sandbox("while True: pass"))

## 4. 통합 Reliability Layer — 모든 도구 호출을 감싸기
회로 차단기 · 재시도 · 멱등성 · 예산을 **하나의 호출 경로**로 묶습니다. 도구 하나가 불안정해도 전체가 안전하게 동작합니다.

In [ ]:
class CircuitBreaker:
    def __init__(self, fail_threshold=3, reset_timeout=0.5):
        self.fail_threshold, self.reset_timeout = fail_threshold, reset_timeout
        self.fails = 0; self.state = "CLOSED"; self.opened_at = 0
    def call(self, fn):
        if self.state == "OPEN":
            if time.time() - self.opened_at >= self.reset_timeout:
                self.state = "HALF_OPEN"
            else:
                raise RuntimeError("🔌 Circuit OPEN — 즉시 차단")
        try:
            r = fn(); self.fails = 0; self.state = "CLOSED"; return r
        except Exception:
            self.fails += 1
            if self.fails >= self.fail_threshold:
                self.state = "OPEN"; self.opened_at = time.time()
            raise

def retry(fn, max_retries=2, base=0.05):
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except Exception:
            if attempt == max_retries:
                raise
            time.sleep(base * (2 ** attempt) * (1 + random.random()))

class ReliableAgent:
    def __init__(self, budget):
        self.budget = budget; self.cb = CircuitBreaker(); self.cache = {}
    def call_tool(self, name, fn, tokens=100, idem_key=None):
        if idem_key and idem_key in self.cache:        # 멱등성: 중복 차단
            print(f"  ♻️ {name}: 캐시 재사용"); return self.cache[idem_key]
        self.budget.charge(tokens)                     # 예산 차감
        result = self.cb.call(lambda: retry(fn))       # 회로차단기(재시도)
        if idem_key:
            self.cache[idem_key] = result
        return result

# 시나리오: 처음 2번 실패 후 성공하는 불안정 도구
calls = {"n": 0}
def flaky():
    calls["n"] += 1
    if calls["n"] <= 2:
        raise ConnectionError("일시적 장애")
    return "OK"

agent = ReliableAgent(Budget(max_tokens=1000, max_steps=10))
print("호출1:", agent.call_tool("flaky", flaky, idem_key="job-1"))          # 재시도로 복구
print("호출2(동일 작업):", agent.call_tool("flaky", flaky, idem_key="job-1"))  # 캐시 → 중복 실행 없음
print(f"총 스텝 {agent.budget.steps}, 토큰 {agent.budget.tokens}, flaky 실제 호출 {calls['n']}회")

## 직접 해보기
회로 차단기가 **OPEN**으로 넘어가는 모습을 확인해 보세요. 항상 실패하는 도구를 `fail_threshold`회 이상 호출하면 이후에는 즉시 차단됩니다.

In [ ]:
# TODO: 항상 예외를 던지는 always_fail 도구를 만들고
#       cb.call(always_fail) 을 4번 호출해 상태가 OPEN 으로 바뀌고 즉시 차단되는지 확인하세요.

In [ ]:
# ▼ 정답 예시 — 직접 해본 뒤 주석을 풀고 실행하세요
# cb = CircuitBreaker(fail_threshold=3)
# def always_fail():
#     raise ConnectionError("계속 실패")
# for i in range(4):
#     try:
#         cb.call(always_fail)
#     except Exception as e:
#         print(f"호출{i+1}: state={cb.state} / {type(e).__name__}: {e}")

## 정리
- **시간/토큰 예산**은 비용·지연 폭주를 사전에 끊습니다(예측 가능한 리소스 소비).
- **샌드박스**는 생성된 코드의 위험(파일·import·무한 루프)을 격리합니다(보안).
- **통합 Reliability Layer**는 회로차단·재시도·멱등성·예산을 한 경로로 묶어 **불안정한 의존성 위에서도 안전한 에이전트**를 만듭니다.